# ROAD MoRF Benchmark

Ноутбук для сравнения `IG`, `NAA` и grid-конфигураций `Cheap-IG` по метрике `ROAD` в режиме `MoRF` на `100` изображениях из `Oxford Pets`.

Бенчмарк classifier-only, использует clean top-1 предсказание `yolo11s-cls` как target-класс, считает `target logit drop AOC` как primary score и сохраняет `top-1 consistency` как secondary диагностику после `Noisy Linear Imputation` на percentiles `10..90`.

## Импорты

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from modules.road_benchmark import benchmark_classifier_road, classifier_method_spec


## Параметры

In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

CLASSIFIER_LAYER = "model.6"
N_STEPS = 128
ROAD_PERCENTILES = [10, 20, 30, 40, 50, 60, 70, 80, 90]
ROAD_NOISE = 0.01
ROAD_NOISE_SEED = 0
CLEAR_EVERY = 8
FD_EPS = 1e-3

CHEAP_IG_SEGMENT_START = 0.0
CHEAP_IG_SEGMENT_END = 0.2
CHEAP_IG_SELECTION_MODE = "positive"
CHEAP_IG_SELECTION_TOP_K_VALUES = [8000, 16000, 32000]

CACHE_ROOT = Path("output/road_cache")
OUTPUT_DIR = Path("output/road_classifier_morf_oxford_pets_100_pred_top1")
REFRESH_CORE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


In [3]:
def collect_oxford_pets_images(image_dir=OXFORD_PETS_DIR, n_images=N_IMAGES):
    image_dir = Path(image_dir)
    if not image_dir.exists():
        raise FileNotFoundError(f"Oxford Pets directory not found: {image_dir}")

    image_paths = []
    for pattern in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        image_paths.extend(image_dir.glob(pattern))

    image_paths = sorted(set(image_paths), key=lambda path: path.name.lower())
    if len(image_paths) < n_images:
        raise ValueError(
            f"Requested {n_images} images, but found only {len(image_paths)} in {image_dir}"
        )
    return [str(path) for path in image_paths[:n_images]]


IMAGE_PATHS = collect_oxford_pets_images()
len(IMAGE_PATHS), IMAGE_PATHS[:5]


(100,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg',
  'oxford_pets/Abyssinian_126.jpg',
  'oxford_pets/Abyssinian_135.jpg'])

## Методы

In [4]:
def build_cheap_ig_variants(fill_mode, fill_rho=None):
    variants = []
    for top_k in CHEAP_IG_SELECTION_TOP_K_VALUES:
        suffix = fill_mode if fill_mode == "zero" else f"{fill_mode}/rho{fill_rho:g}"
        variants.append(
            classifier_method_spec(
                "cheap_ig",
                name=f"Cheap-IG+[0,0.2]/k{top_k}/{suffix}",
                segment_start=CHEAP_IG_SEGMENT_START,
                segment_end=CHEAP_IG_SEGMENT_END,
                selection_mode=CHEAP_IG_SELECTION_MODE,
                selection_top_k=top_k,
                fill_mode=fill_mode,
                fill_rho=fill_rho if fill_rho is not None else 0.8,
            )
        )
    return variants


METHOD_SPECS = [
    classifier_method_spec("ig", name="IG"),
    classifier_method_spec("naa", name="NAA"),
    *build_cheap_ig_variants("zero"),
    *build_cheap_ig_variants("naa_scaled", fill_rho=0.8),
    *build_cheap_ig_variants("naa_scaled", fill_rho=1.0),
]

METHOD_SPECS


[{'kind': 'ig', 'name': 'IG'},
 {'kind': 'naa', 'name': 'NAA'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 8000,
  'fill_mode': 'zero',
  'fill_rho': 0.8,
  'name': 'Cheap-IG+[0,0.2]/k8000/zero'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 16000,
  'fill_mode': 'zero',
  'fill_rho': 0.8,
  'name': 'Cheap-IG+[0,0.2]/k16000/zero'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 32000,
  'fill_mode': 'zero',
  'fill_rho': 0.8,
  'name': 'Cheap-IG+[0,0.2]/k32000/zero'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 8000,
  'fill_mode': 'naa_scaled',
  'fill_rho': 0.8,
  'name': 'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
 

## Запуск Бенчмарка

In [5]:
results = benchmark_classifier_road(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    layer_name=CLASSIFIER_LAYER,
    n_steps=N_STEPS,
    percentiles=ROAD_PERCENTILES,
    noise=ROAD_NOISE,
    noise_seed=ROAD_NOISE_SEED,
    clear_every=CLEAR_EVERY,
    fd_eps=FD_EPS,
    cache_root=CACHE_ROOT,
    target_dir=OUTPUT_DIR,
    save_output=True,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
)

print("output_dir:", results["output_dir"])
print("report_path:", results["report_path"])
print("summary_path:", results["summary_path"])


output_dir: /Users/ashentide/PycharmProjects/PaperImplementations/output/road_classifier_morf_oxford_pets_100_pred_top1
report_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/road_classifier_morf_oxford_pets_100_pred_top1/road_report.md
summary_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/road_classifier_morf_oxford_pets_100_pred_top1/road_summary.json


## Markdown-Отчёт

In [ ]:
display(Markdown(results["report_markdown"]))


# ROAD MoRF Benchmark

- task=`classifier`
- layer_name=`model.6`
- n_steps=128
- n_images=100
- percentiles=[10, 20, 30, 40, 50, 60, 70, 80, 90]
- noise=0.0100
- noise_seed=0
- cache_root=`output/road_cache`

## Aggregate Summary

| Method | Target logit drop AOC | Target logit drop AOC / |clean logit| | Top-1 mean drop | Mean consistency | Mean Rank | Attr Runtime (s) | Eval Runtime (s) | Abs Error |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| IG | 11.7489 +- 3.6832 | 0.8139 +- 0.1824 | 0.9289 +- 0.1117 | 0.0711 +- 0.1117 | 6.7200 +- 4.1547 | 4.4612 +- 0.0650 | 2.1938 +- 0.0577 | 0.7237 +- 0.6076 |
| NAA | 9.9789 +- 3.6076 | 0.6927 +- 0.2057 | 0.8756 +- 0.1971 | 0.1244 +- 0.1971 | 10.1700 +- 2.4210 | 1.9058 +- 0.0399 | 2.1346 +- 0.0550 | 13.1379 +- 3.1544 |
| Cheap-IG+[0,0.2]/k8000/zero | 12.4586 +- 3.2989 | 0.8663 +- 0.1336 | 0.9522 +- 0.0835 | 0.0478 +- 0.0835 | 5.6200 +- 2.9993 | 2.4732 +- 0.0532 | 2.3723 +- 0.0660 | 92.0125 +- 20.2387 |
| Cheap-IG+[0,0.2]/k16000/zero | 12.5045 +- 3.3391 | 0.8698 +- 0.1386 | 0.9478 +- 0.0853 | 0.0522 +- 0.0853 | 5.3700 +- 2.5755 | 2.4670 +- 0.0421 | 2.3682 +- 0.0673 | 96.4712 +- 21.0057 |
| Cheap-IG+[0,0.2]/k32000/zero | 12.5161 +- 3.3629 | 0.8699 +- 0.1383 | 0.9511 +- 0.0806 | 0.0489 +- 0.0806 | 5.2500 +- 2.4955 | 2.4687 +- 0.0415 | 2.3576 +- 0.0718 | 97.9022 +- 21.2363 |
| Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 12.4402 +- 3.3235 | 0.8647 +- 0.1354 | 0.9511 +- 0.0821 | 0.0489 +- 0.0821 | 5.8500 +- 3.0145 | 2.4683 +- 0.0408 | 2.3912 +- 0.0670 | 94.6795 +- 20.7976 |
| Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 12.4926 +- 3.3508 | 0.8689 +- 0.1398 | 0.9489 +- 0.0867 | 0.0511 +- 0.0867 | 5.5000 +- 2.6739 | 2.4669 +- 0.0409 | 2.3678 +- 0.0711 | 97.1905 +- 21.1586 |
| Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 12.5299 +- 3.3667 | 0.8708 +- 0.1378 | 0.9500 +- 0.0807 | 0.0500 +- 0.0807 | 4.9100 +- 2.5811 | 2.4686 +- 0.0412 | 2.3569 +- 0.0730 | 97.9022 +- 21.2363 |
| Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 12.4429 +- 3.3289 | 0.8647 +- 0.1348 | 0.9511 +- 0.0821 | 0.0489 +- 0.0821 | 5.8100 +- 2.9485 | 2.4686 +- 0.0439 | 2.3892 +- 0.0678 | 95.3462 +- 20.9398 |
| Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 12.4985 +- 3.3444 | 0.8693 +- 0.1395 | 0.9500 +- 0.0822 | 0.0500 +- 0.0822 | 5.4700 +- 2.3852 | 2.4703 +- 0.0439 | 2.3655 +- 0.0691 | 97.3697 +- 21.1971 |
| Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 12.5167 +- 3.3754 | 0.8697 +- 0.1382 | 0.9500 +- 0.0822 | 0.0500 +- 0.0822 | 5.3300 +- 2.5497 | 2.4682 +- 0.0439 | 2.3579 +- 0.0728 | 97.9022 +- 21.2363 |

## Core Summary

| Metric | Value |
| --- | ---: |
| clean_top1_logit | 14.3169 +- 2.8821 |
| clean_top1_prob | 0.6515 +- 0.2358 |
| n_pixels_total | 50176.0000 +- 0.0000 |
| n_percentiles | 9.0000 +- 0.0000 |
| core_runtime_s | 0.1909 +- 0.0212 |

## Figures

### road_summary

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_summary.png)

### road_distributions

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_distributions.png)

### road_curves

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_curves.png)

### road_family_envelope

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_family_envelope.png)

### road_cheap_ig_heatmaps

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_cheap_ig_heatmaps.png)

### road_cheap_ig_delta_boxplots

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_cheap_ig_delta_boxplots.png)

### road_pairwise_wins

![](output/road_classifier_morf_oxford_pets_100_pred_top1/figures/road_pairwise_wins.png)

## Per-Image Scores

| Image | IG | NAA | Cheap-IG+[0,0.2]/k8000/zero | Cheap-IG+[0,0.2]/k16000/zero | Cheap-IG+[0,0.2]/k32000/zero | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Abyssinian_1.jpg | 7.0268 | 8.5799 | 9.2133 | 10.1496 | 10.4420 | 9.1247 | 10.2203 | 10.4439 | 9.0752 | 10.0087 | 10.5258 |
| Abyssinian_108.jpg | 8.9423 | 6.3617 | 9.2076 | 9.5924 | 9.5668 | 9.2445 | 9.5733 | 9.5234 | 9.2241 | 9.5228 | 9.5693 |
| Abyssinian_117.jpg | 8.2891 | 5.8852 | 9.1692 | 9.0223 | 9.2498 | 9.0984 | 8.9637 | 9.2667 | 9.2026 | 8.9759 | 9.1075 |
| Abyssinian_126.jpg | 11.8579 | 11.4328 | 11.0671 | 11.1071 | 11.4006 | 11.1646 | 10.9944 | 11.3711 | 10.9039 | 11.0934 | 11.4248 |
| Abyssinian_135.jpg | 3.0207 | 3.3094 | 5.6014 | 5.3073 | 5.3327 | 5.4970 | 5.3757 | 5.3665 | 5.5461 | 5.2870 | 5.4092 |
| Abyssinian_144.jpg | 9.4058 | 7.9759 | 10.0625 | 10.0637 | 10.0416 | 10.0634 | 9.9316 | 10.1307 | 10.0903 | 10.0772 | 10.0350 |
| Abyssinian_154.jpg | 10.5496 | 10.2794 | 9.6929 | 9.7068 | 9.8744 | 9.5498 | 9.6995 | 9.8573 | 9.5203 | 9.7541 | 9.8581 |
| Abyssinian_165.jpg | 10.2825 | 10.9585 | 9.7800 | 10.0233 | 9.9579 | 10.0131 | 10.1734 | 9.9361 | 9.9329 | 10.1607 | 9.8862 |
| Abyssinian_175.jpg | 10.8281 | 9.4015 | 10.4702 | 10.7441 | 10.6962 | 10.6974 | 10.7181 | 10.6601 | 10.6910 | 10.6965 | 10.6782 |
| Abyssinian_184.jpg | 9.7508 | 10.5514 | 10.6086 | 10.7200 | 10.6115 | 10.4973 | 10.6787 | 10.5861 | 10.5732 | 10.6243 | 10.5322 |
| Abyssinian_2.jpg | 12.3898 | 8.6437 | 11.5117 | 11.5885 | 11.5297 | 11.5052 | 11.7269 | 11.7196 | 11.5887 | 11.6620 | 11.6811 |
| Abyssinian_212.jpg | 5.0213 | 5.4975 | 7.6765 | 7.6181 | 7.4926 | 7.7384 | 7.5347 | 7.4573 | 7.6419 | 7.5447 | 7.3933 |
| Abyssinian_224.jpg | 10.0851 | 9.0870 | 10.1611 | 10.2945 | 10.5723 | 10.2259 | 10.3488 | 10.7123 | 10.1104 | 10.4009 | 10.5033 |
| Abyssinian_29.jpg | 10.9563 | 10.8789 | 11.2709 | 11.2149 | 11.4570 | 11.3413 | 11.2477 | 11.4352 | 11.3233 | 11.2905 | 11.4077 |
| Abyssinian_43.jpg | 9.6742 | 8.1674 | 9.5419 | 9.4253 | 9.4748 | 9.4513 | 9.3231 | 9.4135 | 9.4053 | 9.3079 | 9.5559 |
| Abyssinian_52.jpg | 10.1761 | 9.2494 | 9.8159 | 9.5686 | 10.0232 | 9.5247 | 9.3523 | 10.0302 | 9.4432 | 9.5720 | 10.0132 |
| Abyssinian_63.jpg | 11.6230 | 10.6876 | 11.3526 | 11.6654 | 11.4147 | 11.4272 | 11.5552 | 11.5536 | 11.4921 | 11.6550 | 11.3971 |
| Abyssinian_73.jpg | 9.0984 | 5.0530 | 10.0508 | 10.2683 | 10.2719 | 10.0327 | 10.3422 | 10.3073 | 10.0168 | 10.3046 | 10.2684 |
| Abyssinian_83.jpg | 9.0890 | 8.1718 | 9.1049 | 9.3990 | 8.9845 | 9.1831 | 9.3152 | 8.9497 | 9.0834 | 9.4295 | 9.0541 |
| Abyssinian_92.jpg | 7.8942 | 6.8753 | 8.0564 | 8.2426 | 8.2600 | 7.9453 | 8.2627 | 8.2980 | 8.0559 | 8.1590 | 8.2636 |
| american_bulldog_108.jpg | 9.7388 | 5.6910 | 11.0094 | 10.5296 | 10.9752 | 10.8967 | 10.5827 | 10.9635 | 10.8777 | 10.6183 | 10.9701 |
| american_bulldog_117.jpg | 6.6745 | 5.1774 | 6.5146 | 6.4248 | 5.9523 | 6.5375 | 6.3467 | 6.0889 | 6.5552 | 6.3930 | 6.0216 |
| american_bulldog_126.jpg | 9.4385 | 9.5079 | 10.5927 | 10.2218 | 10.2159 | 10.6422 | 10.2200 | 10.2120 | 10.5354 | 10.2379 | 10.1741 |
| american_bulldog_135.jpg | 13.9975 | 12.0125 | 13.5910 | 13.4943 | 13.2759 | 13.5945 | 13.4451 | 13.4090 | 13.5562 | 13.5145 | 13.3061 |
| american_bulldog_144.jpg | 11.9111 | 11.2057 | 9.7722 | 10.1821 | 10.1189 | 9.7169 | 10.1936 | 10.0107 | 9.8902 | 10.1669 | 10.1174 |
| american_bulldog_158.jpg | 11.4955 | 9.7384 | 11.2652 | 11.1963 | 11.5703 | 11.4082 | 11.4043 | 11.6566 | 11.2542 | 11.3513 | 11.6530 |
| american_bulldog_173.jpg | 8.2353 | 6.3908 | 8.7949 | 8.5818 | 8.7056 | 8.9493 | 8.6195 | 8.6225 | 8.8231 | 8.7847 | 8.6588 |
| american_bulldog_182.jpg | 13.2961 | 11.0975 | 13.8087 | 13.6955 | 13.9522 | 13.9161 | 13.6627 | 14.0153 | 13.9088 | 13.7843 | 13.9780 |
| american_bulldog_191.jpg | 12.2578 | 10.8706 | 13.3728 | 13.0065 | 13.2517 | 13.2758 | 12.9976 | 13.1825 | 13.3181 | 13.0080 | 13.2415 |
| american_bulldog_200.jpg | 8.9029 | 7.1446 | 11.2328 | 11.2681 | 11.2136 | 11.1752 | 11.4006 | 11.1566 | 11.2786 | 11.3148 | 11.1486 |
| american_bulldog_212.jpg | 9.5609 | 6.9036 | 9.5511 | 9.5495 | 9.4205 | 9.4703 | 9.4543 | 9.5465 | 9.5818 | 9.4765 | 9.5324 |
| american_bulldog_24.jpg | 10.1008 | 9.9930 | 11.1093 | 11.1254 | 11.1488 | 10.8230 | 10.9389 | 11.1221 | 10.8677 | 10.9726 | 11.1117 |
| american_bulldog_33.jpg | 11.9194 | 11.2676 | 11.3801 | 11.4975 | 11.3252 | 11.2532 | 11.4701 | 11.3677 | 11.2102 | 11.5230 | 11.2807 |
| american_bulldog_43.jpg | 12.1030 | 8.4970 | 13.0813 | 12.9050 | 12.9584 | 13.1099 | 12.9986 | 12.7880 | 13.1706 | 12.9499 | 12.8874 |
| american_bulldog_52.jpg | 3.1058 | 2.4104 | 6.0771 | 6.6404 | 6.0820 | 6.2726 | 6.5803 | 6.2899 | 6.1024 | 6.6793 | 6.0146 |
| american_bulldog_61.jpg | 8.1470 | 8.5588 | 8.1594 | 8.0208 | 8.1136 | 8.0141 | 7.9841 | 8.1146 | 8.0392 | 8.0326 | 8.0191 |
| american_bulldog_70.jpg | 10.4222 | 9.6070 | 10.6738 | 10.3623 | 10.6139 | 10.7277 | 10.3967 | 10.3655 | 10.6630 | 10.3468 | 10.4060 |
| american_bulldog_8.jpg | 11.9289 | 7.9854 | 12.8982 | 13.6438 | 13.5629 | 13.0568 | 13.5693 | 13.5603 | 13.0858 | 13.5888 | 13.5926 |
| american_bulldog_9.jpg | 14.5201 | 11.7987 | 14.2088 | 14.1804 | 14.1794 | 14.0856 | 14.2438 | 14.0205 | 14.1934 | 14.0485 | 14.1501 |
| american_bulldog_99.jpg | 13.5440 | 11.8521 | 13.8328 | 13.1244 | 13.0757 | 13.8458 | 13.1140 | 13.0771 | 13.8427 | 13.1567 | 13.1184 |
| american_pit_bull_terrier_107.jpg | 13.1106 | 11.3347 | 11.7063 | 11.5935 | 11.8471 | 11.8015 | 11.7360 | 11.8832 | 11.7920 | 11.6994 | 11.7220 |
| american_pit_bull_terrier_116.jpg | 10.4997 | 6.2002 | 10.9218 | 11.0598 | 10.6114 | 10.8195 | 11.0673 | 10.5152 | 10.8141 | 10.9528 | 10.5870 |
| american_pit_bull_terrier_125.jpg | 13.6062 | 7.7961 | 12.6755 | 12.8897 | 13.0167 | 12.7962 | 12.7889 | 13.0324 | 12.6991 | 12.8822 | 13.0967 |
| american_pit_bull_terrier_134.jpg | 14.4516 | 14.1374 | 17.0360 | 15.6737 | 15.8810 | 16.6332 | 15.5539 | 15.9995 | 16.6721 | 15.7944 | 15.9264 |
| american_pit_bull_terrier_143.jpg | 15.9409 | 14.2984 | 15.5188 | 15.9205 | 15.8799 | 15.5634 | 15.7928 | 15.8152 | 15.4947 | 15.7372 | 15.8984 |
| american_pit_bull_terrier_152.jpg | 9.6967 | 9.2540 | 10.2257 | 9.8607 | 9.6150 | 9.9655 | 9.7717 | 9.6125 | 9.8825 | 9.9109 | 9.5967 |
| american_pit_bull_terrier_161.jpg | 12.6480 | 10.0947 | 12.5156 | 12.2524 | 12.2246 | 12.3421 | 12.1430 | 12.2399 | 12.3732 | 12.1578 | 12.2282 |
| american_pit_bull_terrier_170.jpg | 14.4225 | 11.6633 | 14.2333 | 14.8048 | 14.5416 | 14.2201 | 14.7853 | 14.7872 | 14.1382 | 14.7620 | 14.5631 |
| american_pit_bull_terrier_18.jpg | 11.6296 | 11.5079 | 10.8716 | 10.9182 | 11.1291 | 10.8914 | 10.9506 | 11.1308 | 10.8683 | 10.9060 | 11.0809 |
| american_pit_bull_terrier_189.jpg | 13.6966 | 8.9544 | 13.2899 | 13.4965 | 13.2609 | 13.1667 | 13.3875 | 13.3600 | 13.2686 | 13.4336 | 13.2807 |
| american_pit_bull_terrier_198.jpg | 11.7362 | 9.9844 | 14.0825 | 14.2865 | 14.6284 | 14.0424 | 14.2722 | 14.5239 | 14.0568 | 14.3521 | 14.4125 |
| american_pit_bull_terrier_22.jpg | 12.3268 | 9.9766 | 11.5639 | 11.4862 | 11.5118 | 11.5669 | 11.5106 | 11.5293 | 11.5580 | 11.4982 | 11.4792 |
| american_pit_bull_terrier_32.jpg | 12.8016 | 9.7027 | 12.3455 | 12.4155 | 12.6874 | 12.3528 | 12.5393 | 12.6956 | 12.3545 | 12.4875 | 12.7290 |
| american_pit_bull_terrier_42.jpg | 13.5026 | 12.2830 | 13.0807 | 12.9092 | 13.1135 | 13.0507 | 13.0151 | 13.1724 | 13.0527 | 13.0728 | 13.1129 |
| american_pit_bull_terrier_51.jpg | 14.3970 | 13.0163 | 13.6180 | 13.2506 | 13.3380 | 13.3764 | 13.1201 | 13.4796 | 13.5426 | 13.2332 | 13.4656 |
| american_pit_bull_terrier_60.jpg | 10.8268 | 13.7448 | 14.3322 | 14.4046 | 14.2636 | 14.1382 | 14.4315 | 14.2731 | 14.2288 | 14.3408 | 14.2707 |
| american_pit_bull_terrier_7.jpg | 12.3800 | 9.2737 | 12.0528 | 11.9672 | 11.6281 | 11.8295 | 11.9206 | 11.7210 | 11.8234 | 11.8990 | 11.5005 |
| american_pit_bull_terrier_79.jpg | 14.6516 | 13.6015 | 14.7464 | 14.1364 | 14.4488 | 14.7287 | 14.0443 | 14.3801 | 14.6274 | 14.0644 | 14.3803 |
| american_pit_bull_terrier_9.jpg | 15.4790 | 13.3285 | 16.3629 | 16.3112 | 16.4254 | 16.4245 | 16.2403 | 16.3941 | 16.3000 | 16.2698 | 16.4808 |
| american_pit_bull_terrier_99.jpg | 9.0042 | 7.1268 | 9.1605 | 8.8491 | 9.4544 | 9.1086 | 9.0055 | 9.3765 | 9.0905 | 8.8593 | 9.2988 |
| basset_hound_107.jpg | 18.5953 | 17.7492 | 18.6585 | 19.2254 | 19.0928 | 18.7157 | 19.1550 | 19.1415 | 18.7891 | 19.1368 | 19.1354 |
| basset_hound_116.jpg | 17.6430 | 17.5322 | 18.4318 | 18.8454 | 19.2200 | 18.5668 | 18.9212 | 19.1949 | 18.4056 | 18.8293 | 19.1488 |
| basset_hound_125.jpg | 15.9624 | 13.8112 | 15.7083 | 15.5314 | 15.4543 | 15.7713 | 15.5683 | 15.6290 | 15.7461 | 15.5566 | 15.5908 |
| basset_hound_134.jpg | 17.3772 | 15.6224 | 17.7538 | 18.0809 | 18.6631 | 17.7106 | 18.0540 | 18.6620 | 17.7748 | 18.1581 | 18.7854 |
| basset_hound_143.jpg | 18.7985 | 15.6463 | 19.1601 | 19.2244 | 19.0879 | 18.9635 | 19.3433 | 19.1046 | 19.0033 | 19.1680 | 19.1296 |
| basset_hound_152.jpg | 19.4397 | 14.7413 | 18.6100 | 19.3521 | 19.3744 | 18.7175 | 19.3613 | 19.3949 | 18.8078 | 19.3604 | 19.3094 |
| basset_hound_161.jpg | 18.6553 | 17.7298 | 18.6341 | 18.6533 | 18.6956 | 18.6816 | 18.7666 | 18.6638 | 18.6628 | 18.7927 | 18.6735 |
| basset_hound_170.jpg | 12.5393 | 13.5412 | 14.7310 | 14.1569 | 14.0094 | 14.8672 | 14.3492 | 14.0206 | 14.7545 | 14.2180 | 14.0406 |
| basset_hound_18.jpg | 11.5262 | 8.5875 | 11.4095 | 11.7558 | 11.9086 | 11.4721 | 11.6930 | 11.8809 | 11.3860 | 11.7770 | 11.8306 |
| basset_hound_189.jpg | 16.7051 | 13.3086 | 16.7010 | 16.9088 | 16.7195 | 17.0141 | 16.9035 | 16.7963 | 17.1068 | 16.8987 | 16.9433 |
| basset_hound_198.jpg | 18.2608 | 15.8845 | 18.1297 | 18.1637 | 18.4735 | 18.1039 | 18.1640 | 18.6558 | 18.1346 | 18.1829 | 18.5647 |
| basset_hound_26.jpg | 21.9591 | 19.3222 | 20.4373 | 20.9634 | 21.0036 | 20.3778 | 20.9408 | 20.9679 | 20.4447 | 20.8887 | 20.9833 |
| basset_hound_35.jpg | 16.6944 | 15.9493 | 17.4873 | 17.9282 | 17.8712 | 17.6768 | 17.9626 | 17.9018 | 17.5722 | 17.9882 | 17.8524 |
| basset_hound_44.jpg | 14.4965 | 15.5543 | 16.5543 | 16.7441 | 16.7406 | 16.7969 | 16.8065 | 16.7529 | 16.7670 | 16.7882 | 16.7233 |
| basset_hound_53.jpg | 13.3773 | 12.4392 | 14.7330 | 14.9970 | 15.1240 | 14.6887 | 14.9567 | 15.2820 | 14.6654 | 15.0424 | 15.1965 |
| basset_hound_62.jpg | 14.6215 | 13.8953 | 13.7780 | 14.0991 | 13.9136 | 13.7417 | 14.0086 | 14.0249 | 13.6670 | 14.0981 | 14.0279 |
| basset_hound_71.jpg | 15.1506 | 12.7376 | 15.6917 | 15.4316 | 15.5553 | 15.6520 | 15.4909 | 15.6398 | 15.5655 | 15.4534 | 15.4683 |
| basset_hound_80.jpg | 17.9549 | 11.2105 | 20.2091 | 19.8822 | 20.0066 | 20.1497 | 19.8726 | 19.9898 | 20.3049 | 19.7159 | 19.9447 |
| basset_hound_9.jpg | 12.4407 | 10.4991 | 13.5463 | 14.2358 | 13.4389 | 13.2289 | 14.1538 | 13.5003 | 13.3407 | 14.0327 | 13.4968 |
| basset_hound_99.jpg | 11.8606 | 8.1507 | 15.9747 | 16.1953 | 16.2201 | 16.1538 | 16.1047 | 16.1996 | 16.1631 | 16.1596 | 16.2610 |
| beagle_108.jpg | 8.8037 | 7.0869 | 11.3287 | 11.1881 | 10.6180 | 11.2910 | 10.9828 | 10.5300 | 11.2326 | 11.1465 | 10.5647 |
| beagle_118.jpg | 8.5034 | 7.0747 | 12.9711 | 12.9796 | 12.8410 | 12.9916 | 13.0188 | 12.8700 | 13.0719 | 13.0630 | 12.9941 |
| beagle_127.jpg | 9.8685 | 8.2449 | 11.5489 | 11.2209 | 11.4475 | 11.6814 | 10.9414 | 11.6369 | 11.5493 | 11.1107 | 11.4819 |
| beagle_137.jpg | 13.6210 | 14.7120 | 12.6681 | 13.2332 | 13.5299 | 12.6388 | 13.2952 | 13.5464 | 12.6443 | 13.2205 | 13.4555 |
| beagle_146.jpg | 14.9413 | 13.5243 | 13.6063 | 13.6837 | 13.5283 | 13.7140 | 13.8453 | 13.5951 | 13.7686 | 13.7973 | 13.6679 |
| beagle_155.jpg | 15.8952 | 13.2199 | 16.4548 | 16.3944 | 16.0589 | 16.5074 | 16.3093 | 16.0484 | 16.4521 | 16.4224 | 16.1079 |
| beagle_165.jpg | 3.5816 | 1.2062 | 4.7892 | 4.8071 | 4.6557 | 4.3592 | 4.8129 | 4.5461 | 4.5388 | 4.7182 | 4.6160 |
| beagle_174.jpg | 4.7343 | 4.9475 | 7.3008 | 7.2680 | 7.7009 | 7.0906 | 7.2254 | 7.6840 | 7.1716 | 7.1323 | 7.7257 |
| beagle_183.jpg | 10.4681 | 6.2251 | 9.3024 | 9.6854 | 8.8046 | 9.4257 | 9.6068 | 8.8855 | 9.2400 | 9.7646 | 8.7967 |
| beagle_192.jpg | 12.8955 | 8.7591 | 14.7065 | 15.5607 | 15.4032 | 14.7423 | 15.5603 | 15.3433 | 14.6674 | 15.5414 | 15.3865 |
| beagle_200.jpg | 10.1593 | 6.9186 | 11.9473 | 12.4110 | 12.2032 | 11.9088 | 12.3937 | 12.0804 | 12.1154 | 12.3370 | 12.0707 |
| beagle_26.jpg | 5.9266 | 4.7454 | 9.2750 | 9.4770 | 9.6313 | 9.0646 | 9.3454 | 9.7249 | 9.1120 | 9.5009 | 9.6604 |
| beagle_35.jpg | 13.7852 | 11.2182 | 14.3136 | 13.9493 | 14.1057 | 14.3734 | 13.9517 | 14.0322 | 14.3274 | 14.0590 | 14.1877 |
| beagle_44.jpg | 7.3048 | 4.2319 | 12.3448 | 12.1358 | 12.2683 | 12.2940 | 12.2567 | 12.3117 | 12.3336 | 12.1570 | 12.2610 |
| beagle_53.jpg | 12.7872 | 9.7787 | 14.3279 | 14.6581 | 14.7612 | 14.2310 | 14.7989 | 14.7091 | 14.2721 | 14.8531 | 14.7433 |
| beagle_62.jpg | 6.2537 | 3.7379 | 10.1497 | 10.0911 | 10.3914 | 10.1557 | 9.9957 | 10.2070 | 10.4517 | 9.9086 | 10.5435 |
| beagle_71.jpg | 14.3818 | 7.7115 | 16.5439 | 16.3206 | 16.1640 | 16.4457 | 16.3807 | 16.0693 | 16.6138 | 16.3650 | 16.1696 |
| beagle_80.jpg | 8.9072 | 4.9725 | 9.4440 | 9.4316 | 9.5188 | 9.5500 | 9.4400 | 9.4690 | 9.5209 | 9.3481 | 9.5267 |
| beagle_9.jpg | 6.7960 | 5.8409 | 12.1497 | 12.3468 | 12.1089 | 12.1356 | 12.1437 | 12.2018 | 12.3489 | 12.0395 | 12.1142 |
| beagle_99.jpg | 7.1501 | 7.7854 | 9.6654 | 9.3710 | 9.4231 | 9.4502 | 9.3373 | 9.5879 | 9.4793 | 9.3354 | 9.4374 |

## Pairwise Win Rate

| Method | IG | NAA | Cheap-IG+[0,0.2]/k8000/zero | Cheap-IG+[0,0.2]/k16000/zero | Cheap-IG+[0,0.2]/k32000/zero | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| IG | — | 0.8700 | 0.3800 | 0.3900 | 0.3700 | 0.3700 | 0.3800 | 0.3800 | 0.3800 | 0.3800 | 0.3800 |
| NAA | 0.1300 | — | 0.0800 | 0.0700 | 0.0700 | 0.1000 | 0.0700 | 0.0700 | 0.0900 | 0.0700 | 0.0800 |
| Cheap-IG+[0,0.2]/k8000/zero | 0.6200 | 0.9200 | — | 0.4500 | 0.4700 | 0.5200 | 0.4600 | 0.4400 | 0.5700 | 0.4600 | 0.4700 |
| Cheap-IG+[0,0.2]/k16000/zero | 0.6100 | 0.9300 | 0.5500 | — | 0.4700 | 0.5500 | 0.5700 | 0.4600 | 0.5100 | 0.5000 | 0.4800 |
| Cheap-IG+[0,0.2]/k32000/zero | 0.6300 | 0.9300 | 0.5300 | 0.5300 | — | 0.5700 | 0.5000 | 0.4600 | 0.5400 | 0.5400 | 0.5200 |
| Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | 0.6300 | 0.9000 | 0.4800 | 0.4500 | 0.4300 | — | 0.4700 | 0.4000 | 0.5200 | 0.4400 | 0.4300 |
| Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | 0.6200 | 0.9300 | 0.5400 | 0.4300 | 0.5000 | 0.5300 | — | 0.4500 | 0.5400 | 0.5000 | 0.4600 |
| Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 | 0.6200 | 0.9300 | 0.5600 | 0.5400 | 0.5400 | 0.6000 | 0.5500 | — | 0.6000 | 0.5800 | 0.5700 |
| Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 0.6200 | 0.9100 | 0.4300 | 0.4900 | 0.4600 | 0.4800 | 0.4600 | 0.4000 | — | 0.4500 | 0.4900 |
| Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 0.6200 | 0.9300 | 0.5400 | 0.5000 | 0.4600 | 0.5600 | 0.5000 | 0.4200 | 0.5500 | — | 0.4500 |
| Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 0.6200 | 0.9200 | 0.5300 | 0.5200 | 0.4800 | 0.5700 | 0.5400 | 0.4300 | 0.5100 | 0.5500 | — |

## Visual Preview

Fixed 5 images shared across all methods; rows are identical across tables.

### IG / NAA

| Raw | IG | NAA |
| --- | --- | --- |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_IG_84a8873b2b.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_NAA_affdbcdc4a.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_IG_84a8873b2b.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_NAA_affdbcdc4a.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_IG_84a8873b2b.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_NAA_affdbcdc4a.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_IG_84a8873b2b.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_NAA_affdbcdc4a.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_IG_84a8873b2b.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_NAA_affdbcdc4a.png) |

### Cheap-IG no tail

| Raw | Cheap-IG+[0,0.2]/k8000/zero | Cheap-IG+[0,0.2]/k16000/zero | Cheap-IG+[0,0.2]/k32000/zero |
| --- | --- | --- | --- |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k8000_zero_8694cb2fb9.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k16000_zero_ad7b9f1a21.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k32000_zero_1160904b06.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k8000_zero_8694cb2fb9.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k16000_zero_ad7b9f1a21.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k32000_zero_1160904b06.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k8000_zero_8694cb2fb9.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k16000_zero_ad7b9f1a21.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k32000_zero_1160904b06.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k8000_zero_8694cb2fb9.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k16000_zero_ad7b9f1a21.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k32000_zero_1160904b06.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k8000_zero_8694cb2fb9.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k16000_zero_ad7b9f1a21.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k32000_zero_1160904b06.png) |

### Cheap-IG + NAA tail (rho=0.8)

| Raw | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho0.8 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho0.8 |
| --- | --- | --- | --- |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k8000_naa_scaled_rho0_8_8788ef9706.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k16000_naa_scaled_rho0_8_aaf4dfea87.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k32000_naa_scaled_rho0_8_00cab69719.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k8000_naa_scaled_rho0_8_8788ef9706.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k16000_naa_scaled_rho0_8_aaf4dfea87.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k32000_naa_scaled_rho0_8_00cab69719.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k8000_naa_scaled_rho0_8_8788ef9706.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k16000_naa_scaled_rho0_8_aaf4dfea87.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k32000_naa_scaled_rho0_8_00cab69719.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k8000_naa_scaled_rho0_8_8788ef9706.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k16000_naa_scaled_rho0_8_aaf4dfea87.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k32000_naa_scaled_rho0_8_00cab69719.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k8000_naa_scaled_rho0_8_8788ef9706.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k16000_naa_scaled_rho0_8_aaf4dfea87.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k32000_naa_scaled_rho0_8_00cab69719.png) |

### Cheap-IG + NAA tail (rho=1.0)

| Raw | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 |
| --- | --- | --- | --- |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k8000_naa_scaled_rho1_8557868e42.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k16000_naa_scaled_rho1_8429ca2bc3.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/01_Abyssinian_1_Cheap_IG_0_0_2_k32000_naa_scaled_rho1_a8d531de6b.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k8000_naa_scaled_rho1_8557868e42.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k16000_naa_scaled_rho1_8429ca2bc3.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/02_Abyssinian_108_Cheap_IG_0_0_2_k32000_naa_scaled_rho1_a8d531de6b.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k8000_naa_scaled_rho1_8557868e42.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k16000_naa_scaled_rho1_8429ca2bc3.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/03_Abyssinian_117_Cheap_IG_0_0_2_k32000_naa_scaled_rho1_a8d531de6b.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k8000_naa_scaled_rho1_8557868e42.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k16000_naa_scaled_rho1_8429ca2bc3.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/04_Abyssinian_126_Cheap_IG_0_0_2_k32000_naa_scaled_rho1_a8d531de6b.png) |
| ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_raw.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k8000_naa_scaled_rho1_8557868e42.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k16000_naa_scaled_rho1_8429ca2bc3.png) | ![](output/road_classifier_morf_oxford_pets_100_pred_top1/preview/05_Abyssinian_135_Cheap_IG_0_0_2_k32000_naa_scaled_rho1_a8d531de6b.png) |

: 